# 03 · Linhas de base: regras e TF-IDF

Treina TF-IDF + Regressão Logística / Random Forest no **treino humano do RePro** e avalia no teste. Se os rótulos do LLM já existirem (notebook 04), treina também com eles, nos mesmos textos.

In [ ]:
# Configuração: no Colab, clona o repositório, instala o pacote `absa`
# e guarda cache do LLM, modelos e predições no Drive (sobrevive à queda da sessão)
import sys
if "google.colab" in sys.modules:
    !git clone -q https://github.com/vnevesl/pln-absa-olist.git 2>/dev/null || git -C pln-absa-olist pull -q
    %cd pln-absa-olist
    !pip -q install -e "."
    from google.colab import drive
    drive.mount("/content/drive")
    !mkdir -p /content/drive/MyDrive/pln-absa-olist/llm_cache /content/drive/MyDrive/pln-absa-olist/models /content/drive/MyDrive/pln-absa-olist/interim
    !ln -sfn /content/drive/MyDrive/pln-absa-olist/llm_cache data/llm_cache
    !ln -sfn /content/drive/MyDrive/pln-absa-olist/models models
    !ln -sfn /content/drive/MyDrive/pln-absa-olist/interim data/interim

In [ ]:
import pandas as pd
from absa.config import INTERIM, LABEL_COLUMNS
from absa.heuristics import heuristic_labels
from absa.metrics import bootstrap_ci, per_aspect_report, tuple_f1
from absa.models.baseline import TfidfAspectClassifier
from absa.preprocess import clean_classic
from absa.repro import load_repro
treino, val, teste, _ = load_repro()
print(len(treino), len(val), len(teste))

In [ ]:
def salvar(nome, pred):
    INTERIM.mkdir(parents=True, exist_ok=True)
    pred.assign(review_id=teste.review_id.values).to_csv(INTERIM / f"pred_repro_teste_{nome}.csv", index=False)
    f1, lo, hi = bootstrap_ci(teste, pred)
    print(f"{nome}: F1 macro {f1:.3f} (IC95% {lo:.3f}–{hi:.3f}) | F1 tupla {tuple_f1(teste, pred):.3f}")

## Regras (palavras-chave)

In [ ]:
regras = pd.DataFrame([heuristic_labels(t) for t in teste.texto])
regras.columns = LABEL_COLUMNS
salvar("regras", regras)
per_aspect_report(teste, regras).round(3)

## TF-IDF com rótulos humanos

In [ ]:
X_treino = treino.texto.map(clean_classic)
X_teste = teste.texto.map(clean_classic)
modelos = {}
for nome in ["logreg", "rf"]:
    modelos[nome] = TfidfAspectClassifier(nome).fit(X_treino, treino)
    salvar(f"tfidf_{nome}_humano", modelos[nome].predict(X_teste))
per_aspect_report(teste, modelos["logreg"].predict(X_teste)).round(3)

In [ ]:
# Termos mais associados a "entrega negativo" (explicabilidade global)
modelos["logreg"].top_terms("entrega", "negativo")

## TF-IDF com rótulos do LLM (mesmos textos)

In [ ]:
arq = INTERIM / "repro_treino_rotulos_llm.csv"
if arq.exists():
    fracos = pd.read_csv(arq)
    clf = TfidfAspectClassifier("logreg").fit(fracos.texto.map(clean_classic), fracos)
    salvar("tfidf_logreg_llm", clf.predict(X_teste))
else:
    print("rode o notebook 04 primeiro")